# Candidate fitting and model selection

Run this notebook only after the setup/smoke-test stage has placed a pretrained
IResNet-100 checkpoint at the configured local-only path. It performs ten fits:
three Triplet Margin, four ArcFace, and three SupCon. No selected candidate is
retrained on validation data.

In [ ]:
# Configuration (the only values an operator should need to review or edit).
from pathlib import Path

REPOSITORY_ROOT = next(
    root for root in (Path.cwd().resolve(), Path.cwd().resolve().parent)
    if (root / "GLOSSARY.md").exists() and (root / "face_name_metric_learning").exists()
)
MANIFEST_PATH = REPOSITORY_ROOT / "data" / "processed" / "manifest.csv"
HDF5_PATH = REPOSITORY_ROOT / "data" / "processed" / "faces.h5"
PRETRAINED_CHECKPOINT_PATH = (
    REPOSITORY_ROOT / "models" / "pretrained" / "iresnet100.pth"
)
TRIAL_RESULTS_PATH = (
    REPOSITORY_ROOT / "artifacts" / "face_name_metric_learning" / "trial_results.csv"
)
EXPECTED_SPLIT_COUNTS = {"train": 45_078, "validation": 9_660, "test": 9_660}
EXPECTED_CLASS_COUNT = 100


In [ ]:
import sys

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import torch
import pytorch_metric_learning

from face_name_metric_learning import (
    DEFAULT_TRAINING_CONFIG,
    candidate_configurations,
    read_manifest,
    validate_manifest_hdf5,
    validate_trial_ledger,
)
from face_name_metric_learning.pytorch_training import train_and_select_candidates

candidates = candidate_configurations()
assert len(candidates) == 10
assert [candidate.loss for candidate in candidates].count("triplet_margin") == 3
assert [candidate.loss for candidate in candidates].count("arcface") == 4
assert [candidate.loss for candidate in candidates].count("supcon") == 3
assert torch.cuda.is_available(), "A CUDA device is required; training is not run on CPU."
assert PRETRAINED_CHECKPOINT_PATH.is_file(), (
    f"Pretrained checkpoint missing: {PRETRAINED_CHECKPOINT_PATH}. "
    "Run the setup stage or copy the approved IResNet-100 checkpoint locally."
)
device = torch.device("cuda")
print({
    "device": str(device),
    "gpu": torch.cuda.get_device_name(device),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "pytorch_metric_learning": pytorch_metric_learning.__version__,
})

In [ ]:
data_contract = validate_manifest_hdf5(
    MANIFEST_PATH,
    HDF5_PATH,
    expected_split_counts=EXPECTED_SPLIT_COUNTS,
    expected_class_count=EXPECTED_CLASS_COUNT,
)
manifest_rows = read_manifest(MANIFEST_PATH)
assert data_contract["split_counts"] == EXPECTED_SPLIT_COUNTS
assert data_contract["class_count"] == EXPECTED_CLASS_COUNT
print(data_contract)

The fitting loop samples 16 names by 8 distinct images. A pre-training CUDA
memory probe retries only as an 8-by-8 controlled fallback. Each epoch uses full
batches and records the actual steps and image draws. Triplet Margin chooses
uniform in-batch positives/negatives; ArcFace owns a temporary 100-class head;
SupCon uses the library's same-name positive construction.

In [ ]:
trial_rows = train_and_select_candidates(
    repository_root=REPOSITORY_ROOT,
    manifest_path=MANIFEST_PATH,
    hdf5_path=HDF5_PATH,
    trial_results_path=TRIAL_RESULTS_PATH,
    pretrained_checkpoint_path=PRETRAINED_CHECKPOINT_PATH,
    manifest_rows=manifest_rows,
    config=DEFAULT_TRAINING_CONFIG,
)
ledger = validate_trial_ledger(TRIAL_RESULTS_PATH)
assert len(ledger["rows"]) == 10
ledger["selected_trials"]

In [ ]:
display_columns = [
    "trial_id",
    "loss",
    "hyperparameters_json",
    "epochs_attempted",
    "best_epoch",
    "best_validation_triplet_accuracy",
    "notes",
    "selected",
]
selected = [
    {column: row[column] for column in display_columns}
    for row in sorted(
        trial_rows,
        key=lambda row: (row["loss"], -float(row["best_validation_triplet_accuracy"])),
    )
]
selected

In [ ]:
from face_name_metric_learning import trial_checkpoint_path

checkpoint_paths = [trial_checkpoint_path(REPOSITORY_ROOT, row) for row in trial_rows]
assert all(path.is_file() for path in checkpoint_paths)
assert all("models/metric_learning" in path.as_posix() for path in checkpoint_paths)
print("Best checkpoints:")
for path in checkpoint_paths:
    print(path.relative_to(REPOSITORY_ROOT))